# Распознавание типа космического корабля (Zero-Shot Classification)
В этом ноутбуке мы используем модель CLIP для анализа скачанных фотографий ракет и предсказания их типа без предварительного обучения на нашем датасете.

In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
print("✅ Библиотеки импортированы")

In [ ]:
# Определяем пути
if os.path.exists('/opt/airflow'):
    BASE_DIR = '/opt/airflow'
else:
    BASE_DIR = os.getcwd()

IMAGES_DIR = os.path.join(BASE_DIR, 'data', 'images')
OUTPUT_FILE = os.path.join(BASE_DIR, 'data', 'ml_predictions.csv')

print(f"🖼️ Images dir: {IMAGES_DIR}")
print(f"📄 Output: {OUTPUT_FILE}")
os.makedirs(IMAGES_DIR, exist_ok=True)

In [ ]:
image_files = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print(f"Найдено изображений: {len(image_files)}")
if len(image_files)==0:
    raise Exception("Нет изображений! Запустите DAG в Airflow.")

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
ROCKET_LABELS = ["Falcon 9 rocket","Falcon Heavy rocket","Soyuz rocket","Ariane 5 rocket","Electron rocket","Long March rocket","Atlas V rocket","Delta IV rocket","Antares rocket","Starship rocket","Space Shuttle","launch vehicle"]

print("Загрузка модели CLIP...")
model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Модель на {device}")

In [ ]:
def classify_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(text=ROCKET_LABELS, images=image, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            probs = model(**inputs).logits_per_image.softmax(dim=1)[0]
        top_prob, top_idx = probs.max(dim=0)
        return {"image_name": os.path.basename(image_path), "predicted_rocket": ROCKET_LABELS[top_idx.item()], "confidence": round(top_prob.item()*100,2), "error": None}
    except Exception as e:
        return {"image_name": os.path.basename(image_path), "predicted_rocket": None, "confidence": 0, "error": str(e)}

In [ ]:
results = []
for img_path in image_files:
    res = classify_image(img_path)
    results.append(res)

In [ ]:
results = []
for i, img in enumerate(image_files,1):
    print(f"[{i}/{len(image_files)}] {os.path.basename(img)}...", end=" ")
    res = classify_image(img)
    results.append(res)
    if res["error"]: print(f"❌ {res['error']}")
    else: print(f"✅ {res['predicted_rocket']} ({res['confidence']}%)")
df = pd.DataFrame(results)
df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Сохранено в {OUTPUT_FILE}")

In [ ]:
print(df[['image_name','predicted_rocket','confidence']].head())